# core

> Basic jevey objects

In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| exports
import os
from fastcore.all import *

TYPESAFE_API_URL = "https://api.typesafe.ai/"

In [ ]:
#| hide
from cachy import enable_cachy, doms
enable_cachy(doms=doms+(TYPESAFE_API_URL,))

# Primitives

Jev has 3 Primitives: `Choice`, `Score` and `Noul`. All these objects have a name and instructions.



In [ ]:
#| exports
class Prim(AttrDict):
    """Base class for Noul, Choice and Score"""
    def __init__(self, type: str, name: str, instructions: str):
        store_attr()

In [ ]:
prim = Prim(type="base", name="test", instructions="Some instructions")
prim

```python
{'instructions': 'Some instructions', 'name': 'test', 'type': 'base'}
```

In [ ]:
#| exports
class Noul(Prim):
    """Bool decision value"""
    def __init__(self, name: str, instructions: dict):
        super().__init__(type="noul", name=name, instructions=instructions)

In [ ]:
n = Noul("is_bug", "Is the customer reporting a software defect?")
n 

```python
{ 'instructions': 'Is the customer reporting a software defect?',
  'name': 'is_bug',
  'type': 'noul'}
```

# Client

For the `Client` make sure to set up `JEV_API_KEY` as an env variable.

In [ ]:
#| exports
import json
from fastspec.spec import SpecParser
from fastspec.oapi import OpenAPIClient

In [ ]:
#| exports
spec_obj = dict2obj(json.loads(urlread(f"{TYPESAFE_API_URL}openapi.json")))
spec_obj.servers = [{"url": TYPESAFE_API_URL}]
spec_obj.servers

[{'url': 'https://api.typesafe.ai/'}]

In [ ]:
#| exports
JEV_SPEC = SpecParser.from_openapi(spec_obj)
JEV_SPEC

SpecParser(base_url='https://api.typesafe.ai/', ops=2)

In [ ]:
#| exports
class Client(BasicRepr):
    """Client for the Jev API"""
    HEADERS = {"Authorization": f"Bearer {os.environ['JEV_API_KEY']}",
    "Content-Type": "application/json"}
    def __init__(self, model="jev-latest"):
        store_attr()
        self.c = OpenAPIClient(JEV_SPEC, headers=self.HEADERS)
        self.questions = dict()


In [ ]:
c = Client()
c

Client(model='jev-latest', c=<fastspec.oapi.OpenAPIClient object>, questions={})

In [ ]:
#| exports
@patch
def __iadd__(self:Client, q:Prim): 
    self.questions[q.name] = {k: v for k, v in q.items() if k != "name"}
    return self

In [ ]:
c += n
c

Client(model='jev-latest', c=<fastspec.oapi.OpenAPIClient object>, questions={'is_bug': {'type': 'noul', 'instructions': 'Is the customer reporting a software defect?'}})

In [ ]:
#| exports
@patch
def __call__(self:Client, state:str):
        return self.c.func_dict["/v1/systemone:POST"](
            model=self.model, state=state, questions=self.questions,
        )

In [ ]:
res = await c("This is working perfectly")
res

```python
{ 'answers': {'is_bug': {'noul': 0.03, 'type': 'noul'}},
  'model': 'jev-1.13.0',
  'usage': {'input_tokens': 278, 'output_tokens': 21}}
```

In [ ]:
#| exports
class Decision:
    pass

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()